# 📘 Guide 07: AI-assisted coding and debugging

**Who it's for:** you use (or want to use) Cursor, Claude Code or Copilot, and would like it to help more and mislead less. **Time:** about 20 minutes.

AI assistants are superb pair programmers for this course: they explain errors, write boilerplate and suggest tests. They also make confident mistakes. The skill is in giving them context and checking their work.

### 🎯 You'll learn
1. What to put in a prompt to get a useful answer
2. Asking for explanations, not just fixes
3. A "trust but verify" loop with small tests
4. Where assistants commonly go wrong in this course

## Part 1: Give it context

A vague prompt gets a vague answer. Compare:

> ❌ *"My code doesn't work, fix it."*

> ✅ *"In `week2/day3.ipynb` I call `retriever.invoke(question)` and get `KeyError: 'source'` (full traceback below). I'm using LangChain 1.x and Chroma. Explain the cause first, then suggest the smallest fix. Don't change unrelated code."*

The good prompt includes: **where** (file, cell), **what happened** (the full error), **versions** (LangChain 1.x!), and **what kind of help** you want.

In Cursor, `@`-mention files (e.g. `@gradewise/schemas/exam.py`) so the assistant reads the real code instead of guessing. The repo's `AGENTS.md` tells assistants the course conventions; it's worth reading yourself too.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">LLMs were trained on lots of <b>old LangChain code</b>. If an assistant suggests <code>create_react_agent</code>, <code>LLMChain</code> or <code>initialize_agent</code>, that's pre-1.0 style. Ask it to use LangChain 1.x (<code>create_agent</code>, <code>init_chat_model</code>) and check against the docs.</span>
        </td>
    </tr>
</table>

## Part 2: Ask for understanding

You'll learn far more by asking *why* than by pasting fixes. Useful prompts:
- *"Explain this cell line by line, as if to a CS201 student."*
- *"What would happen if `chunk_overlap` were 0 here?"*
- *"What are two alternative ways to write this, and their trade-offs?"*

## Part 3: Trust, but verify

Here's a function an assistant might write when asked to "scale marks to a percentage and cap at 100". It looks right. Is it?

In [ ]:
def to_percentage(got, out_of):
    return min(100, got / out_of * 100)


# 💡 A few quick checks catch most mistakes, including the ones nobody thought about
checks = [(4, 5, 80.0), (5, 5, 100.0), (6, 5, 100.0), (0, 5, 0.0)]
for got, out_of, expected in checks:
    result = to_percentage(got, out_of)
    print(f"{'✅' if result == expected else '❌'} to_percentage({got}, {out_of}) = {result} (expected {expected})")

try:
    to_percentage(3, 0)
except ZeroDivisionError:
    print("❌ to_percentage(3, 0) crashes: what should a question worth 0 marks do?")

The happy cases pass, but the edge case (`out_of=0`, or negative marks) wasn't handled. Asking the assistant *"What edge cases does this miss? Write tests for them."* would have surfaced it. Here's a sturdier version:

In [ ]:
def to_percentage(got: float, out_of: float) -> float:
    if out_of <= 0:
        raise ValueError(f"out_of must be positive, got {out_of}")
    return max(0.0, min(100.0, got / out_of * 100))


print(to_percentage(6, 5), to_percentage(-1, 5))

## Part 4: Debugging with an assistant: a recipe

1. **Reproduce** the error and copy the *full* traceback (not just the last line).
2. Paste it with the cell's code and say what you expected.
3. Ask for the **cause** first. Only then a fix.
4. Apply the fix, **re-run** (Restart + Run All for notebooks), and run the tests: `uv run python -m pytest -q`.
5. If it's still broken, tell the assistant what changed. Don't let it go in circles: after two failed attempts, step back and use Guide 08's method.

## Part 5: Common traps in this course

| Assistant suggests… | Why it's a problem | Ask instead for… |
|---|---|---|
| Hard-coding `"gpt-4o"` in code | model names live in `.env` | `gradewise.models.chat_model()` |
| `open(path)` without encoding | breaks on Windows with non-ASCII text | `Path.read_text(encoding="utf-8")` |
| `uv run pytest` | blocked on some Windows machines | `uv run python -m pytest` |
| Putting the API key in code | leaks your key | `.env` + `load_dotenv` |

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Try it</h2>
            <span style="color:#f59e0b;">Pick any cell from <code>week1/day1.ipynb</code> and ask your assistant to explain it line by line. Then ask it to write three pytest tests for <code>to_percentage</code> above, including edge cases. Run them. Did any of its tests have a mistake?</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/resources.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#3b82f6;">Resources</h2>
            <span style="color:#3b82f6;">• Cursor documentation: <a href="https://docs.cursor.com/">docs.cursor.com/</a><br/>• Claude Code documentation: <a href="https://docs.claude.com/en/docs/claude-code/overview">docs.claude.com/en/docs/claude-code/overview</a><br/>• LangChain docs (check the version!): <a href="https://docs.langchain.com/">docs.langchain.com/</a></span>
        </td>
    </tr>
</table>